<a href="https://colab.research.google.com/github/odellus/colab/blob/main/GenX_Colab_Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# <img src="https://raw.githubusercontent.com/GenXProject/GenX.jl/refs/heads/main/docs/src/assets/logo_readme.svg" height=50/> _Colab Notebook_

## Instructions
1. Work on a copy of this notebook: _File_ > _Save a copy in Drive_ (you will need a Google account). Alternatively, you can download the notebook using _File_ > _Download .ipynb_, then upload it to [Colab](https://colab.research.google.com/).
2. If you need a GPU: _Runtime_ > _Change runtime type_ > _Harware accelerator_ = _GPU_.
3. Execute the following cell (click on it and press Ctrl+Enter) to install Julia, IJulia and other packages, in our case GenX (if needed, update `JULIA_VERSION` and the other parameters). This takes a couple of minutes.
4. Reload this page (press Ctrl+R, or ⌘+R, or the F5 key) and continue to the next section.

_Notes_:
* If your Colab Runtime gets reset (e.g., due to inactivity), repeat steps 2, 3 and 4.
* After installation, if you want to change the Julia version or activate/deactivate the GPU, you will need to reset the Runtime: _Runtime_ > _Factory reset runtime_ and repeat steps 3 and 4.

# Install IJulia and GenX

We are going to use IJulia to turn our google colab notebook that defaults to IPython into an IJulia notebook. This will let us run the GenX package in the cloud without having to install it locally.

In [18]:
# @title Install IJulia and GenX
%%shell
set -e

#---------------------------------------------------#
JULIA_VERSION="1.8.2" # any version ≥ 0.7.0
JULIA_PACKAGES="IJulia GenX"
JULIA_PACKAGES_IF_GPU="CUDA" # or CuArrays for older Julia versions
JULIA_NUM_THREADS=2
#---------------------------------------------------#

if [ -z `which julia` ]; then
  # Install Julia
  JULIA_VER=`cut -d '.' -f -2 <<< "$JULIA_VERSION"`
  echo "Installing Julia $JULIA_VERSION on the current Colab Runtime..."
  BASE_URL="https://julialang-s3.julialang.org/bin/linux/x64"
  URL="$BASE_URL/$JULIA_VER/julia-$JULIA_VERSION-linux-x86_64.tar.gz"
  wget -nv $URL -O /tmp/julia.tar.gz # -nv means "not verbose"
  tar -x -f /tmp/julia.tar.gz -C /usr/local --strip-components 1
  rm /tmp/julia.tar.gz

  # Install Packages
  nvidia-smi -L &> /dev/null && export GPU=1 || export GPU=0
  if [ $GPU -eq 1 ]; then
    JULIA_PACKAGES="$JULIA_PACKAGES $JULIA_PACKAGES_IF_GPU"
  fi
  for PKG in `echo $JULIA_PACKAGES`; do
    echo "Installing Julia package $PKG..."
    julia -e 'using Pkg; pkg"add '$PKG'; precompile;"' &> /dev/null
  done
  # Install kernel and rename it to "julia"
  echo "Installing IJulia kernel..."
  julia -e 'using IJulia; IJulia.installkernel("julia", env=Dict(
      "JULIA_NUM_THREADS"=>"'"$JULIA_NUM_THREADS"'"))'
  KERNEL_DIR=`julia -e "using IJulia; print(IJulia.kerneldir())"`
  KERNEL_NAME=`ls -d "$KERNEL_DIR"/julia*`
  mv -f $KERNEL_NAME "$KERNEL_DIR"/julia

  echo ''
  echo "Successfully installed `julia -v`!"
  echo "Please reload this page (press Ctrl+R, ⌘+R, or the F5 key) then"
  echo "jump to the 'Checking the Installation' section."

fi

Unrecognized magic `%%shell`.

Julia does not use the IPython `%magic` syntax.   To interact with the IJulia kernel, use `IJulia.somefunction(...)`, for example.  Julia macros, string macros, and functions can be used to accomplish most of the other functionalities of IPython magics.


# Checking the Installation
The `versioninfo()` function should print your Julia version and some other info about the system:

In [1]:
versioninfo()

Julia Version 1.8.2
Commit 36034abf260 (2022-09-29 15:21 UTC)
Platform Info:
  OS: Linux (x86_64-linux-gnu)
  CPU: 2 × Intel(R) Xeon(R) CPU @ 2.20GHz
  WORD_SIZE: 64
  LIBM: libopenlibm
  LLVM: libLLVM-13.0.1 (ORCJIT, broadwell)
  Threads: 1 on 2 virtual cores
Environment:
  LD_LIBRARY_PATH = /usr/local/nvidia/lib:/usr/local/nvidia/lib64
  JULIA_NUM_THREADS = 


In [17]:
try
    using CUDA
catch
    println("No GPU found.")
else
    run(`nvidia-smi`)
    # Create a new random matrix directly on the GPU:
    M_on_gpu = CUDA.CURAND.rand(2^11, 2^11)
    @btime $M_on_gpu * $M_on_gpu; nothing
end

No GPU found.


# Run GenX
We're going to walk through running a single example of the immensely popular GenX electricity resource exapansion model.

In [2]:
using GenX

  ____           __  __   _ _
 / ___| ___ _ __ \ \/ /  (_) |
| |  _ / _ \ '_ \ \  /   | | |
| |_| |  __/ | | |/  \ _ | | |
 \____|\___|_| |_/_/\_(_)/ |_|
                       |__/
 Version: 0.4.1


In [6]:
; git clone https://github.com/GenXProject/GenX.jl

Cloning into 'GenX.jl'...


In [12]:
; cd GenX.jl/example_systems/1_three_zones

/content/GenX.jl/example_systems/1_three_zones


In [13]:
include("Run.jl")


Configuring Settings
Clustering Time Series Data (Grouped)...
Reading Input CSV Files
Network.csv Successfully Read!
Demand (load) data Successfully Read!
Fuels_data.csv Successfully Read!


[ Info: Thermal.csv Successfully Read.
[ Info: Vre.csv Successfully Read.
[ Info: Storage.csv Successfully Read.
[ Info: Resource_minimum_capacity_requirement.csv Successfully Read.



Summary of resources loaded into the model:
-------------------------------------------------------
	Resource type 		Number of resources
	Thermal        		3
	VRE            		4
	Storage        		3
Total number of resources: 10
-------------------------------------------------------
Generators_variability.csv Successfully Read!
Validating time basis
Minimum_capacity_requirement.csv Successfully Read!
CO2_cap.csv Successfully Read!
CSV Files Successfully Read In From /content/GenX.jl/example_systems/1_three_zones
Configuring Solver
Loading Inputs
Reading Input CSV Files
Network.csv Successfully Read!
Demand (load) data Successfully Read!
Fuels_data.csv Successfully Read!

Summary of resources loaded into the model:
-------------------------------------------------------
	Resource type 		Number of resources
	Thermal        		3
	VRE            		4
	Storage        		3
Total number of resources: 10
-------------------------------------------------------
Generators_variability.csv Successful

[ Info: Thermal.csv Successfully Read.
[ Info: Vre.csv Successfully Read.
[ Info: Storage.csv Successfully Read.
[ Info: Resource_minimum_capacity_requirement.csv Successfully Read.


Discharge Module
Non-served Energy Module
Investment Discharge Module
Unit Commitment Module
Fuel Module
CO2 Module
Investment Transmission Module
Transmission Module
Dispatchable Resources Module
Storage Resources Module
Storage Investment Module
Storage Core Resources Module
Storage Resources with Symmetric Charge/Discharge Capacity Module
Thermal (Unit Commitment) Resources Module
CO2 Policies Module
Minimum Capacity Requirement Module
Time elapsed for model building is
21.153620306
Solving Model
Running HiGHS 1.8.1 (git hash: 4a7f24ac6): Copyright (c) 2024 HiGHS under MIT licence terms
Coefficient ranges:
  Matrix [4e-07, 1e+01]
  Cost   [1e-04, 6e+02]
  Bound  [0e+00, 0e+00]
  RHS    [2e-03, 4e+03]
Presolving model
118107 rows, 81152 cols, 467034 nonzeros  0s
107345 rows, 70390 cols, 456090 nonzeros  0s
Presolve : Reductions: rows 107345(-46053); columns 70390(-49748); elements 456090(-59349)
Solving the presolved LP
IPX model has 107345 rows, 70390 columns and 456090 nonzeros
Inp

...

Time elapsed for writing fuel consumption is
1.515677912


[ Info: Writing Full Time Series for Emissions Plant


Time elapsed for writing co2 is
0.339382096
Time elapsed for writing price is
0.345047039


[ Info: Writing Full Time Series for Price


Time elapsed for writing energy revenue is
1.010063811
Time elapsed for writing charging cost is
0.534946054
Time elapsed for writing subsidy is
1.053575565
Time elapsed for writing time weights is
0.195734253
Time elapsed for writing co2 cap is
0.357118445
Time elapsed for writing minimum capacity requirement is
0.315348993
Time elapsed for writing net revenue is
3.223958181
Wrote outputs to /content/GenX.jl/example_systems/1_three_zones/results
Time elapsed for writing is
34.886932891


In [19]:
; ls results

capacity.csv
capacityfactor.csv
charge.csv
ChargingCost.csv
CO2_prices_and_penalties.csv
commit.csv
costs.csv
curtail.csv
emissions.csv
emissions_plant.csv
EnergyRevenue.csv
flow.csv
FuelConsumption_plant_MMBTU.csv
FuelConsumption_total_MMBTU.csv
Fuel_cost_plant.csv
Full_TimeSeries
MinCapReq_prices_and_penalties.csv
NetRevenue.csv
network_expansion.csv
nse.csv
power_balance.csv
power.csv
prices.csv
RegSubsidyRevenue.csv
reliability.csv
run_settings.yml
shutdown.csv
start.csv
status.csv
storagebal_duals.csv
storage.csv
SubsidyRevenue.csv
system_summary.yml
time_weights.csv
tlosses.csv
